# SEN12MS-CR 학습 노트북

이 노트북은 `main.py`와 같은 흐름으로 학습을 돌리되, 중간 결과와 복원 예시를 Colab/Jupyter에서 바로 보기 쉽게 정리한 버전이다.

- 실행 환경 준비
- 학습 설정값 지정
- 모델/옵티마이저 훅 연결
- 배치 shape 확인
- 학습 로그 표/그래프 확인
- 복원 결과 예시 저장 및 시각화


In [ ]:
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
WORKDIR = Path.cwd()

# 노트북에서는 uv를 먼저 설치한 뒤 현재 커널에 프로젝트를 바로 설치한다.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)

CR_TRAIN_REF = "c4386724646b55a8c9629b9016bc18d11ad1dc86"

if (WORKDIR / "pyproject.toml").exists():
    subprocess.run(
        [
            "uv",
            "pip",
            "install",
            "--system",
            "--no-cache",
            "--reinstall-package",
            "datasets",
            "--reinstall-package",
            "fsspec",
            "-e",
            ".",
        ],
        check=True,
    )
else:
    subprocess.run(
        [
            "uv",
            "pip",
            "install",
            "--system",
            "--no-cache",
            "--reinstall-package",
            "datasets",
            "--reinstall-package",
            "fsspec",
            f"git+https://github.com/smturtle2/cr-train.git@{CR_TRAIN_REF}",
        ],
        check=True,
    )

if IN_COLAB and Path("/content/drive/MyDrive").exists():
    CHECKPOINT_DIR = Path("/content/drive/MyDrive/cr-project/checkpoints")
else:
    CHECKPOINT_DIR = Path("/content/checkpoints") if IN_COLAB else WORKDIR / "checkpoints"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print(f"IN_COLAB={IN_COLAB}")
print(f"WORKDIR={WORKDIR}")
print(f"CHECKPOINT_DIR={CHECKPOINT_DIR}")

In [ ]:
import random

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from torch import nn

from cr_train import MAE, Trainer, TrainerConfig, build_sen12mscr_loaders

RGB_CHANNELS = (3, 2, 1)

# 처음에는 작은 설정으로 전체 파이프라인이 도는지 확인하는 쪽이 낫다.
batch_size = 8
seed = 42
split = "official"
io_profile = "smooth"
max_epochs = 10
train_max_samples = 2048
val_max_samples = 512
test_max_samples = 512
resume = None
run_test = False
num_examples = 4
example_stage = "val"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device={device}")

In [ ]:
def seed_everything(seed: int) -> None:
    # 실험 재현성을 위해 파이썬, 넘파이, 파이토치 시드를 함께 고정한다.
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def build_model() -> nn.Module:
    # Trainer는 model(sar, cloudy)를 호출하므로 입력 시그니처를 맞춰 구현한다.
    raise NotImplementedError("프로젝트 모델로 build_model()을 교체하세요.")


def build_optimizer(model: nn.Module) -> torch.optim.Optimizer:
    # build_model()에서 만든 파라미터와 함께 사용할 optimizer를 정의한다.
    raise NotImplementedError("프로젝트 optimizer로 build_optimizer()를 교체하세요.")


def build_criterion() -> nn.Module:
    return nn.L1Loss()


def build_metrics() -> list[nn.Module]:
    return [MAE()]


def flatten_record(record: dict) -> dict[str, int | float]:
    # Trainer 로그를 한 줄짜리 dict로 펴서 표와 그래프에 바로 쓴다.
    row = {
        "epoch": int(record["epoch"]),
        "global_step": int(record["global_step"]),
    }
    for stage in ("train", "val", "test"):
        for name, value in record.get(stage, {}).items():
            row[f"{stage}_{name}"] = float(value)
    return row


def format_row(row: dict[str, int | float]) -> str:
    metric_keys = sorted(key for key in row if key not in {"epoch", "global_step"})
    parts = [f"epoch={row['epoch']}", f"global_step={row['global_step']}"]
    parts.extend(f"{key}={row[key]:.4f}" for key in metric_keys)
    return " | ".join(parts)


def style_frame(frame: pd.DataFrame, caption: str):
    metric_cols = [col for col in frame.columns if col not in {"epoch", "global_step"}]
    style = frame.style.hide(axis="index").set_caption(caption)
    return style.format({col: "{:.4f}" for col in metric_cols})


def plot_history_df(history_df: pd.DataFrame) -> None:
    import matplotlib.pyplot as plt

    metric_cols = [col for col in history_df.columns if col not in {"epoch", "global_step"}]
    loss_cols = [col for col in metric_cols if col.endswith("_loss")]
    other_cols = [col for col in metric_cols if col not in loss_cols]
    groups = [keys for keys in (loss_cols, other_cols) if keys]

    fig, axes = plt.subplots(len(groups), 1, figsize=(10, 4 * len(groups)), sharex=True)
    if len(groups) == 1:
        axes = [axes]

    epochs = history_df["epoch"]
    colors = plt.get_cmap("tab10")

    for ax, keys in zip(axes, groups):
        for index, key in enumerate(keys):
            ax.plot(
                epochs,
                history_df[key],
                label=key,
                color=colors(index % 10),
                linewidth=2.2,
                marker="o",
                markersize=5,
            )
        ax.grid(True, alpha=0.25)
        ax.legend(frameon=False)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.set_ylabel("value")

    axes[0].set_title("training history")
    axes[-1].set_xlabel("epoch")
    plt.show()


def show_history(history: list[dict[str, int | float]]) -> pd.DataFrame:
    history_df = pd.DataFrame(history)
    display(style_frame(history_df, "history"))
    plot_history_df(history_df)

    best_key = "val_loss" if "val_loss" in history_df.columns else "train_loss"
    best_row = history_df.loc[[history_df[best_key].idxmin()]]
    display(style_frame(best_row, f"best row by {best_key}"))
    return history_df


def show_test_metrics(metrics: dict[str, float]) -> pd.DataFrame:
    import matplotlib.pyplot as plt

    test_df = pd.DataFrame([{f"test_{name}": float(value) for name, value in metrics.items()}])
    display(style_frame(test_df, "test metrics"))

    series = test_df.iloc[0].sort_values()
    fig, ax = plt.subplots(figsize=(8, max(2.5, 0.7 * len(series))))
    ax.barh(series.index, series.values, color="#4C78A8")
    ax.set_title("test metrics")
    ax.set_xlabel("value")
    ax.grid(True, axis="x", alpha=0.25)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.show()
    return test_df


def select_rgb(image: torch.Tensor) -> torch.Tensor:
    # 13채널 optical 입력은 B4, B3, B2 순서를 RGB 미리보기로 사용한다.
    return image[list(RGB_CHANNELS)]


def normalize_rgb_triplet(
    cloudy: torch.Tensor,
    prediction: torch.Tensor,
    target: torch.Tensor,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    rgb_tensors = [select_rgb(tensor).detach().cpu().float() for tensor in (cloudy, prediction, target)]
    merged = np.concatenate(
        [tensor.permute(1, 2, 0).reshape(-1, 3).numpy() for tensor in rgb_tensors],
        axis=0,
    )
    low = np.quantile(merged, 0.02, axis=0)
    high = np.quantile(merged, 0.98, axis=0)
    scale = np.maximum(high - low, 1e-6)

    images = []
    for tensor in rgb_tensors:
        image = tensor.permute(1, 2, 0).numpy()
        image = np.clip((image - low) / scale, 0.0, 1.0)
        images.append(image ** (1.0 / 2.2))
    return images[0], images[1], images[2]


def normalize_map(image: torch.Tensor) -> np.ndarray:
    array = image.detach().cpu().float().numpy()
    low, high = np.quantile(array, [0.02, 0.98])
    return np.clip((array - low) / max(high - low, 1e-6), 0.0, 1.0)


def show_restoration_examples(
    model: nn.Module,
    dataloader,
    device: torch.device,
    num_examples: int,
    stage: str,
) -> None:
    # 학습 후 cloudy / prediction / target / SAR / error를 같이 본다.
    import matplotlib.pyplot as plt

    output_dir = CHECKPOINT_DIR / "examples"
    output_dir.mkdir(parents=True, exist_ok=True)

    was_training = model.training
    model.eval()
    shown = 0

    iterator = iter(dataloader)
    try:
        with torch.no_grad():
            while shown < num_examples:
                try:
                    batch = next(iterator)
                except StopIteration:
                    break

                sar, cloudy = batch["inputs"]
                target = batch["target"]
                metadata = batch["metadata"]
                prediction = model(sar.to(device), cloudy.to(device)).cpu()

                for index in range(prediction.shape[0]):
                    cloudy_rgb, prediction_rgb, target_rgb = normalize_rgb_triplet(
                        cloudy[index],
                        prediction[index],
                        target[index],
                    )
                    sar_map = normalize_map(sar[index].mean(dim=0))
                    error_map = normalize_map((prediction[index] - target[index]).abs().mean(dim=0))

                    title = (
                        f"{stage} example {shown + 1} | "
                        f"{metadata['season'][index]}/scene_{metadata['scene'][index]}/patch_{metadata['patch'][index]}"
                    )

                    fig, axes = plt.subplots(1, 5, figsize=(18, 4))
                    fig.suptitle(title, fontsize=11)
                    panels = (
                        ("Cloudy RGB", cloudy_rgb, None),
                        ("Prediction RGB", prediction_rgb, None),
                        ("Target RGB", target_rgb, None),
                        ("SAR Mean", sar_map, "gray"),
                        ("Abs Error", error_map, "magma"),
                    )

                    for ax, (panel_title, image, cmap) in zip(axes, panels):
                        if cmap is None:
                            ax.imshow(image)
                        else:
                            ax.imshow(image, cmap=cmap)
                        ax.set_title(panel_title)
                        ax.axis("off")

                    fig.tight_layout()
                    fig.subplots_adjust(top=0.80)
                    path = output_dir / f"{stage}_example_{shown + 1:02d}.png"
                    fig.savefig(path, dpi=180, bbox_inches="tight")
                    plt.show()
                    plt.close(fig)
                    print(f"saved: {path}")

                    shown += 1
                    if shown == num_examples:
                        break
    finally:
        del iterator
        model.train(was_training)


In [ ]:
seed_everything(seed)

train_loader, val_loader, test_loader = build_sen12mscr_loaders(
    batch_size=batch_size,
    seed=seed,
    split=split,
    io_profile=io_profile,
)

# 모델을 건드리기 전에 배치 shape부터 확인해 두면 디버깅이 훨씬 쉽다.
sample_iterator = iter(train_loader)
try:
    sample_batch = next(sample_iterator)
finally:
    del sample_iterator
sar, cloudy = sample_batch["inputs"]
target = sample_batch["target"]

print("sar   :", tuple(sar.shape), sar.dtype)
print("cloudy:", tuple(cloudy.shape), cloudy.dtype)
print("target:", tuple(target.shape), target.dtype)
print("metadata keys:", list(sample_batch["metadata"].keys()))

In [ ]:
model = build_model().to(device)
optimizer = build_optimizer(model)
criterion = build_criterion()
metrics = build_metrics()

trainer = Trainer(
    model=model,
    optimizer=optimizer,
    criterion=criterion,
    metrics=metrics,
    config=TrainerConfig(
        max_epochs=max_epochs,
        train_max_samples=train_max_samples,
        val_max_samples=val_max_samples,
        test_max_samples=test_max_samples,
        checkpoint_dir=CHECKPOINT_DIR,
    ),
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
)

if resume is not None:
    trainer.load_checkpoint(resume)

trainer

In [ ]:
history = []

for record in trainer.step():
    row = flatten_record(record)
    history.append(row)
    print(format_row(row))

history_df = show_history(history)

if run_test:
    test_metrics = trainer.test()
    test_df = show_test_metrics(test_metrics)
else:
    test_df = None

example_loader = val_loader if example_stage == "val" else test_loader
show_restoration_examples(
    model=model,
    dataloader=example_loader,
    device=device,
    num_examples=num_examples,
    stage=example_stage,
)